# NUTDTS 816 Time Series Analysis
## L04 Decomposition II: transformations, STL, adjustment, features

Lab notebook for Chapter 2 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### Carried forward from Lab 3 (run these cells first; they define the objects used below)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import tsdata
ap   = tsdata.airpassengers()     # multiplicative
grid = tsdata.nigeria_grid()      # additive (simulated)
a10  = tsdata.a10()               # monthly antidiabetic drug sales, Australia: multiplicative, strong trend
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
ap.plot(ax=axes[0], title='Airline passengers: multiplicative')
grid.plot(ax=axes[1], title='Grid generation: additive')
a10.plot(ax=axes[2], title='a10 drug sales: multiplicative, growing amplitude')
for ax in axes: ax.set_xlabel('')
_caption = 'Two multiplicative series and one additive series.'

In [ ]:
def centred_2xm_ma(s, m):
    """2 x m moving average (for even m) or m-MA (odd m)."""
    if m % 2 == 1:
        return s.rolling(m, center=True).mean()
    first = s.rolling(m).mean()                 # trailing m-MA
    return first.rolling(2).mean().shift(-m // 2)  # average adjacent pairs and recentre

t_hat = centred_2xm_ma(ap, 12)
ax = ap.plot(figsize=(8, 3.2), label='observed', lw=1)
t_hat.plot(ax=ax, color='#B8860B', lw=2, label='2×12-MA trend estimate')
ax.set_title('Airline passengers with a 2×12-MA trend-cycle estimate'); ax.legend(); ax.set_xlabel('')
print('First valid trend value:', t_hat.first_valid_index().date(), '| last:', t_hat.last_valid_index().date())
_caption = 'The moving average removes the annual pattern and most of the noise. Six months are lost at each end.'

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
dec_add  = seasonal_decompose(grid, model='additive', period=12)
dec_mult = seasonal_decompose(ap,   model='multiplicative', period=12)

fig, axes = plt.subplots(4, 2, figsize=(11, 8), sharex='col')
for col, (dec, name) in enumerate([(dec_add, 'Grid generation (additive)'), (dec_mult, 'Airline passengers (multiplicative)')]):
    for row, comp in enumerate(['observed', 'trend', 'seasonal', 'resid']):
        getattr(dec, comp).plot(ax=axes[row, col], lw=1); axes[row, col].set_ylabel(comp); axes[row, col].set_xlabel('')
    axes[0, col].set_title(name)
_caption = 'Classical decomposition panels. Note the gaps at the ends of trend and remainder, and the identical seasonal pattern every year.'

In [ ]:
print('Seasonal factors, airline (multiplicative), one year:')
print(dec_mult.seasonal['1950'].round(3).to_string())
print('\nSeasonal effects, grid (additive, MW), one year:')
print(dec_add.seasonal['2020'].round(0).to_string())

## Decomposition II: transformations, STL, adjustment, features

### 2.4 Transformations

In [ ]:
from scipy import stats
lam = stats.boxcox_normmax(a10.values, method='mle')
print(f'MLE Box-Cox lambda for a10: {lam:.3f}  (0 = log)')
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
a10.plot(ax=axes[0], title='a10: original (λ = 1)')
np.log(a10).plot(ax=axes[1], title='a10: log (λ = 0)')
pd.Series(stats.boxcox(a10.values, lmbda=lam), index=a10.index).plot(ax=axes[2], title=f'a10: Box-Cox (λ = {lam:.2f})')
for ax in axes: ax.set_xlabel('')
_caption = 'The log already makes the seasonal amplitude nearly constant; the fitted λ is close to zero and adds little.'

### 2.5 STL: Seasonal-Trend decomposition using LOESS

In [ ]:
from statsmodels.tsa.seasonal import STL
stl = STL(np.log(ap), period=12, seasonal=13, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(9, 7)
_caption = 'STL decomposition of log airline passengers. Trend and remainder extend to both ends; the seasonal pattern is allowed to evolve.'

In [ ]:
g_out = grid.copy(); g_out['2021-03-01'] -= 900   # inject a large one-month collapse
cls = seasonal_decompose(g_out, model='additive', period=12)
stl_r = STL(g_out, period=12, seasonal=13, robust=True).fit()
stl_n = STL(g_out, period=12, seasonal=13, robust=False).fit()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
cls.trend.plot(ax=axes[0], label='classical 2×12-MA'); stl_n.trend.plot(ax=axes[0], label='STL (not robust)'); stl_r.trend.plot(ax=axes[0], label='STL (robust)')
axes[0].set_title('Trend estimates around an injected outlier (Mar 2021)'); axes[0].legend(); axes[0].set_xlim(pd.Timestamp('2019-06-01'), pd.Timestamp('2023-01-01')); axes[0].set_xlabel('')
cls.seasonal['2022'].plot(ax=axes[1], label='classical'); stl_r.seasonal['2022'].plot(ax=axes[1], label='STL robust'); axes[1].set_title('Seasonal component, 2022'); axes[1].legend(); axes[1].set_xlabel('')
_caption = 'The outlier drags the classical moving average down for a year; robust STL barely notices it.'

In [ ]:
# How much does the seasonal pattern evolve? Compare the STL seasonal component across years
seas = stl.seasonal
piv = pd.DataFrame({'v': seas.values, 'year': seas.index.year, 'month': seas.index.month}).pivot(index='month', columns='year', values='v')
ax = piv.plot(colormap='viridis', legend=False, figsize=(8, 3), title='STL seasonal component of log passengers, one line per year')
ax.set_xlabel('Month'); ax.set_xticks(range(1, 13))
_caption = 'The seasonal pattern in log passengers sharpens over the 1950s: the summer peak grows relative to the troughs.'

### 2.6 Seasonal adjustment and detrending

In [ ]:
stl_cpi = STL(np.log(tsdata.nigeria_cpi()), period=12, seasonal=13, robust=True).fit()
cpi = tsdata.nigeria_cpi()
cpi_sa = np.exp(np.log(cpi) - stl_cpi.seasonal)
infl_raw = 100 * np.log(cpi).diff()
infl_sa  = 100 * (np.log(cpi) - stl_cpi.seasonal).diff()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
infl_raw['2021':].plot(ax=axes[0], label='raw', lw=1); infl_sa['2021':].plot(ax=axes[0], label='seasonally adjusted', lw=1.5)
axes[0].set_title('Monthly inflation (% log change), CPI (simulated)'); axes[0].legend(); axes[0].set_xlabel('')
(100 * stl_cpi.seasonal['2025']).plot(ax=axes[1], title='Estimated seasonal effect on log CPI, 2025 (%)'); axes[1].set_xlabel('')
_caption = 'Seasonal adjustment removes the regular within-year pattern from monthly inflation, leaving the trend-plus-noise that policy analysis needs.'

### 2.7 Time series features: strength of trend and seasonality

In [ ]:
def strength_features(s, period, log=False):
    y = np.log(s) if log else s
    r = STL(y, period=period, seasonal=13, robust=True).fit()
    ft = max(0, 1 - r.resid.var() / (r.trend + r.resid).var())
    fs = max(0, 1 - r.resid.var() / (r.seasonal + r.resid).var())
    return round(ft, 3), round(fs, 3)

rows = []
for name, s, lg in [('Airline passengers', ap, True), ('a10 drug sales', a10, True), ('Grid generation (sim.)', grid, False),
                    ('CPI (sim.)', cpi, True), ('NGN/USD (sim.)', tsdata.nigeria_fx(), True), ('Monthly inflation (sim.)', infl_raw.dropna(), False),
                    ('White noise', pd.Series(np.random.default_rng(3).normal(size=144), index=ap.index), False)]:
    ft, fs = strength_features(s, 12, lg); rows.append((name, ft, fs))
print(pd.DataFrame(rows, columns=['series', 'trend strength', 'seasonal strength']).to_string(index=False))

### 2.8 Worked example: decomposing the a10 series end to end

In [ ]:
y = np.log(a10)
res = STL(y, period=12, seasonal=7, robust=True).fit()
fig = res.plot(); fig.set_size_inches(9, 7)
_caption = 'STL of log antidiabetic drug sales. The trend rises steadily and flattens only at the very end of the sample; the seasonal component grows; the remainder is small apart from a few January spikes.'

In [ ]:
sa = np.exp(y - res.seasonal)
ax = a10.plot(figsize=(8, 3.2), lw=1, label='observed')
sa.plot(ax=ax, lw=1.5, color='#B8860B', label='seasonally adjusted')
np.exp(res.trend).plot(ax=ax, lw=1.5, color='#555555', label='trend')
ax.set_title('a10: observed, seasonally adjusted and trend'); ax.legend(); ax.set_xlabel('')
print('Seasonal factors (multiplicative, last year):'); print(np.exp(res.seasonal['2007-07':'2008-06']).round(3).to_string())
_caption = 'The seasonal factors show a January peak about 30% above trend (Australian patients stockpiling before the subsidy threshold resets) and a February trough.'

## Exercises

4. A forecast on the log scale has mean 6.0 and variance 0.09 at horizon 12. Compute the median and the mean forecast on the original scale and the percentage difference between them.
5. The `h02` series (cortecosteroid drug sales) is available in the course data module. Decide on a transformation, decompose, and report the strength-of-trend and strength-of-seasonality features. Compare them with `a10`.
6. Explain to a planning officer, in three sentences and without jargon, the difference between the seasonally adjusted inflation rate and the headline rate, and when each should be quoted.

In [ ]:
# Your work here
